In [1]:
import numpy as np
from scipy.stats import qmc

from alb_sim.config.run import RunConfig
from alb_sim.config.sea_floor import SeaFloorConfig
from alb_sim.config.sea_surface import SeaSurfaceConfig
from alb_sim.config.sensor import SensorConfig
from alb_sim.config.simulation import SimulationConfig
from alb_sim.config.water import TurbidityLayerConfig, WaterConfig
from alb_sim.execution.parallel import run_parallel
from alb_sim.plotting.plot_stacked_waveform import plot_stacked_waveform
from alb_sim.plotting.plot_waveform import plot_waveform

Define the parameter space by selecting parameters for consideration and setting the value ranges.

In [2]:
param_ranges = {
    "height": (0.5, 7.5),
    "absorption": (0.005, 0.05),
    "scattering": (1.0, 5.0),
    "floor_albedo": (0.05, 0.3),
    "surface_roughness": (0.01, 0.1),
    "field_of_view": (5e-3, 40e-3),
    "aperture_radius": (0.001, 0.2),
}


def build_simulation_config(params: dict):
    return SimulationConfig(
        water=WaterConfig(
            layers=(
                TurbidityLayerConfig(
                    height=params["height"],
                    absorption_coefficient=params["absorption"],
                    scattering_coefficient=params["scattering"],
                ),
            )
        ),
        sensor=SensorConfig(
            field_of_view=params["field_of_view"],
            aperture_radius=params["aperture_radius"],
        ),
        sea_floor=SeaFloorConfig(albedo=params["floor_albedo"]),
        sea_surface=SeaSurfaceConfig(roughness=params["surface_roughness"]),
    )

Generate Latin Hypercube samples.

In [ ]:
n_samples = 5
d = len(param_ranges)

# generate samples in [0, 1) intervals
sampler = qmc.LatinHypercube(d=d)
sample = sampler.random(n=n_samples)

# extract lower and upper bounds
l_bounds = [v[0] for v in param_ranges.values()]
u_bounds = [v[1] for v in param_ranges.values()]

scaled_sample = qmc.scale(sample, l_bounds, u_bounds)

param_names = list(param_ranges.keys())

samples = [dict(zip(param_names, row)) for row in scaled_sample]

Visualize the top n samples, default is n = 5.

In [ ]:
import pandas as pd

# Show first n samples
pd.DataFrame(samples).head(n=5)

Define RunConfig that fits to the given hardware.

In [7]:
run_config = RunConfig(
    processes=8,
    batches_forward=8 * 1,
    batches_backward=8 * 1,
)

Run the simulation given the parameter samples.

In [ ]:
results = []

for params in samples:

    simulation_config = build_simulation_config(params)

    waveform, _, _ = run_parallel(simulation_config, run_config)

    results.append({"params": params, "waveform": waveform})

Visualize the results. Results can also be saved automatically by passing the `save_path` argument.

In [ ]:
from IPython.display import Markdown, display

for result in results:
    display(Markdown("---"))
    display(pd.DataFrame([result["params"]]).head())

    plot_stacked_waveform(result["waveform"])

    plot_waveform(np.sum(list(result["waveform"].values()), axis=0))